### **Notebook 1**

# Group 2: Luca Milani, Marta Laskowska, Monika Kaczorowska
## Hackathon - webscraping failed

### Libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import math
import requests
import json
import time
from datetime import datetime as DateTime, timezone, timedelta
import time
import bs4 as bs
import yfinance as yf
import datetime
from scipy.stats import norm
from collections import Counter
import re
from urllib.parse import quote_plus


from ipywidgets import interact, IntSlider, Checkbox
from functools import lru_cache
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option('display.max_colwidth', None)

## Task #2 : Social media coverage of press campaigns
- Use names of targeted firms and NGO + date of the campaign to see if social
 media talked about the story (Twitter, Reddit, Facebook, etc.)
- Requires translation + sentiment analysis

In [2]:
df = pd.read_csv('events_date_ngos_company_wide.csv')
df.head()

,date,isin_corporate_name_cleaned,company_parent_country,company_parent,target_country1,target_country2,target_country3,target_country4,target_country5,target_country6,ngo_name1,ngo_name2,ngo_name3,ngo_name4,ngo_name5
0,2010-12-23,Deutsche Bank,Germany,Deutsche Bank,Germany,NaN,NaN,NaN,NaN,NaN,Urgewald,NaN,NaN,NaN,NaN
1,2010-12-23,Commerzbank,Germany,Commerzbank,Germany,NaN,NaN,NaN,NaN,NaN,Urgewald,NaN,NaN,NaN,NaN
2,2010-12-23,UniCredit,Italy,UniCredit,Germany,NaN,NaN,NaN,NaN,NaN,Urgewald,NaN,NaN,NaN,NaN
3,2010-12-23,UniCredit,Italy,UniCredit,Germany,NaN,NaN,NaN,NaN,NaN,Urgewald,NaN,NaN,NaN,NaN
4,2010-12-22,NaN,Sweden,Skandinaviska Enskilda Banken AB (SEB),Canada,NaN,NaN,NaN,NaN,NaN,Greenpeace Norway,NaN,NaN,NaN,NaN


In [ ]:
# GDELT Task #1: Events ➜ 10-day window news scrape (bank-required), with per-event metrics
#
# What this does
# 1) Reads your CSV with columns:
#    - isin_corporate_name_cleaned, company_name, date, ngo_name1..ngo_name5
# 2) Builds per-row queries for GDELT DOC API within [event_date, event_date+10 days]
# 3) Enforces: the article MUST mention the bank (isin_corporate_name_cleaned or company_name)
#    NGO mention is optional (we *try* bank+NGO queries first, then fall back to bank-only)
# 4) Aggregates per-event metrics (counts, domains, languages, top titles, sample URLs)
# 5) Flags covered_by_major_outlet if a domain is in your curated list
# 6) Saves two CSV files:
#       - /mnt/data/gdelt_event_metrics.csv   (one row per input event)
#       - /mnt/data/gdelt_raw_hits.csv        (one row per article hit, for audit)
#
# Notes
# - Internet access is required to actually call GDELT. This notebook shows a complete, runnable pipeline.
# - Customize MAJOR_OUTLETS and INPUT_CSV before running.
#
# If you want me to run it on your CSV here, upload the file and rerun.


# =========================
# 🔧 CONFIG
# =========================
INPUT_CSV = "events_date_ngos_company_wide.csv"   # <-- replace with your actual CSV path
DATE_COL = "date"
BANK_COL_1 = "isin_corporate_name_cleaned"
BANK_COL_2 = "company_parent"
NGO_COLS = [f"ngo_name{i}" for i in range(1, 6)]

# Curated list of "major outlets" — add/remove domains to fit your research
MAJOR_OUTLETS = {
    # Global
    "reuters.com","bloomberg.com","ft.com","wsj.com","nytimes.com","apnews.com","theguardian.com","bbc.co.uk","bbc.com",
    # Europe
    "lemonde.fr","elpais.com","faz.net","handelsblatt.com","repubblica.it","corriere.it","ansa.it","elconfidencial.com",
    # US finance/tech
    "cnbc.com","forbes.com","marketwatch.com","seekingalpha.com","techcrunch.com",
    # 🇬🇧 UK – broadsheets & business
    "telegraph.co.uk",        # The Telegraph
    "thetimes.co.uk",         # The Times
    "independent.co.uk",      # The Independent
    "standard.co.uk",         # Evening Standard
    "inews.co.uk",            # The i Paper
    "economist.com",          # The Economist (magazine, but major)
    "cityam.com",             # City A.M. (business)
    "sky.com",                # Sky (news.sky.com subdomain often appears)
    "itv.com",                # ITV News (itv.com/news)
    "channel4.com",    
}

# GDELT Doc API settings
GDELT_ENDPOINT = "https://api.gdeltproject.org/api/v2/doc/doc"
MAXRECORDS = 250        # API max per call
SORT = "DateDesc"       # newest first
MODE = "ArtList"        # list of articles
PAUSE_SEC = 1.0         # polite rate limiting between calls

# How many top items to keep in summary fields
TOP_N = 5

In [10]:
# =========================
# Helper functions
# =========================

def to_gdelt_datetime(dt):
    """GDELT expects YYYYMMDDHHMMSS — we default to day start/end for the window."""
    return dt.strftime("%Y%m%d%H%M%S")

def normalize_str(x):
    return str(x).strip() if pd.notna(x) else ""

def build_query_variants(bank_names, ngo_names):
    """
    Build a list of query strings to try, in priority order:
      1) "bank" "ngo" (for each ngo)
      2) "bank" (bank-only)
    We always include at least one bank in the query (bank is required by your spec).
    """
    queries = []
    clean_banks = [q for q in {normalize_str(b) for b in bank_names} if q]
    clean_ngos = [q for q in {normalize_str(n) for n in ngo_names} if q]

    # bank + ngo combos first
    for b in clean_banks:
        for n in clean_ngos:
            queries.append(f'"{b}" "{n}"')

    # fallbacks: bank-only
    for b in clean_banks:
        queries.append(f'"{b}"')

    # Deduplicate while preserving order
    seen = set()
    dedup = []
    for q in queries:
        if q not in seen:
            dedup.append(q)
            seen.add(q)
    return dedup

def call_gdelt(query, start_dt, end_dt):
    """Call the GDELT Doc API and return list of article dicts (or empty list)."""
    params = {
        "query": query,
        "mode": MODE,
        "format": "json",
        "maxrecords": MAXRECORDS,
        "sort": SORT,
        "startdatetime": to_gdelt_datetime(start_dt),
        "enddatetime": to_gdelt_datetime(end_dt),
    }
    try:
        resp = requests.get(GDELT_ENDPOINT, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        return data.get("articles", []) or []
    except Exception as e:
        # In production you might log this
        return []

def contains_bank(text, bank_candidates):
    """Check if any bank candidate is mentioned in the text (case-insensitive)."""
    t = text.lower()
    for b in bank_candidates:
        b = b.lower()
        if b and b in t:
            return True
    return False

def extract_domain(url):
    try:
        # Simple domain extraction
        m = re.search(r"https?://([^/]+)/", url + "/")
        if m:
            return m.group(1).lower()
        return ""
    except Exception:
        return ""

def summarize_articles(articles, bank_candidates):
    """Return metrics dict and a DataFrame of cleaned article rows."""
    if not articles:
        return {
            "n_articles": 0,
            "n_articles_with_bank": 0,
            "domains_top": "",
            "languages_top": "",
            "top_titles": "",
            "covered_by_major_outlet": 0,
            "sample_urls": ""
        }, pd.DataFrame(columns=["seendate","domain","language","title","url"])

    # Normalize to DataFrame
    rows = []
    for a in articles:
        rows.append({
            "seendate": a.get("seendate"),
            "domain": (a.get("domain") or "").lower(),
            "language": a.get("language"),
            "title": a.get("title") or "",
            "url": a.get("url") or "",
        })
    df = pd.DataFrame(rows).drop_duplicates(subset=["url"]).reset_index(drop=True)

    # Require bank presence (title or URL) per your spec
    df["has_bank"] = df.apply(
        lambda r: contains_bank((r.get("title") or "") + " " + (r.get("url") or ""), bank_candidates), axis=1
    )
    df_with_bank = df[df["has_bank"]].copy()

    # Aggregations
    n_articles = len(df)
    n_articles_with_bank = len(df_with_bank)

    # Top domains / languages among bank-filtered results (fallback to all if none)
    base = df_with_bank if n_articles_with_bank > 0 else df
    domains_top = "; ".join(base["domain"].value_counts().head(TOP_N).index.tolist())
    languages_top = "; ".join(base["language"].value_counts().head(TOP_N).index.astype(str).tolist())
    top_titles = "; ".join(base["title"].head(TOP_N).tolist())
    sample_urls = "; ".join(base["url"].head(TOP_N).tolist())

    covered_by_major = int(any(d in MAJOR_OUTLETS for d in base["domain"].unique()))

    metrics = {
        "n_articles": n_articles,
        "n_articles_with_bank": n_articles_with_bank,
        "domains_top": domains_top,
        "languages_top": languages_top,
        "top_titles": top_titles,
        "covered_by_major_outlet": covered_by_major,
        "sample_urls": sample_urls
    }
    return metrics, df

In [11]:
# =========================
# Main pipeline
# =========================

def run_gdelt_pipeline(input_csv=INPUT_CSV):
    # Read input
    events = pd.read_csv(input_csv)
    # normalize / coerce date
    events[DATE_COL] = pd.to_datetime(events[DATE_COL], errors="coerce")
    # primary bank name candidates per row
    # we will treat both columns as possible bank mentions
    events[BANK_COL_1] = events[BANK_COL_1].fillna("")
    events[BANK_COL_2] = events[BANK_COL_2].fillna("")

    # Build NGO list per row
    for c in NGO_COLS:
        if c not in events.columns:
            events[c] = np.nan

    # Output collectors
    metrics_rows = []
    raw_hits_rows = []

    for idx, row in events.iterrows():
        event_date = row[DATE_COL]
        if pd.isna(event_date):
            continue
        start_dt = pd.Timestamp(event_date).normalize()
        end_dt = start_dt + timedelta(days=10)

        bank_candidates = [normalize_str(row[BANK_COL_1]), normalize_str(row[BANK_COL_2])]
        ngo_candidates = [normalize_str(row[c]) for c in NGO_COLS]

        # Build prioritized query variants (bank+ngo, then bank-only)
        queries = build_query_variants(bank_candidates, ngo_candidates)

        # Accumulate results across variants, but de-dup by URL
        all_articles = []
        seen_urls = set()

        for q in queries:
            arts = call_gdelt(q, start_dt, end_dt)
            for a in arts:
                url = a.get("url")
                if url and url not in seen_urls:
                    all_articles.append(a)
                    seen_urls.add(url)
            time.sleep(PAUSE_SEC)  # polite rate limit

        # Summarize
        metrics, df_hits = summarize_articles(all_articles, bank_candidates)

        # Save per-event metrics row
        metrics_rows.append({
            "event_row": idx,
            "event_date": start_dt.date().isoformat(),
            "end_date": end_dt.date().isoformat(),
            "bank_name_1": bank_candidates[0],
            "bank_name_2": bank_candidates[1],
            "ngos": "; ".join([n for n in ngo_candidates if n]),
            "queries_tried": " | ".join(queries),
            **metrics,
        })

        # Save raw hits for audit
        if not df_hits.empty:
            df_hits["event_row"] = idx
            df_hits["event_date"] = start_dt.date().isoformat()
            df_hits["bank_candidates"] = "; ".join([b for b in bank_candidates if b])
            df_hits["ngo_candidates"] = "; ".join([n for n in ngo_candidates if n])
            raw_hits_rows.append(df_hits)

    # Combine and write out
    metrics_df = pd.DataFrame(metrics_rows)
    raw_hits_df = pd.concat(raw_hits_rows, ignore_index=True) if raw_hits_rows else pd.DataFrame(
        columns=["seendate","domain","language","title","url","event_row","event_date","bank_candidates","ngo_candidates"]
    )

    metrics_path = "/mnt/data/gdelt_event_metrics.csv"
    raw_path = "/mnt/data/gdelt_raw_hits.csv"
    metrics_df.to_csv(metrics_path, index=False)
    raw_hits_df.to_csv(raw_path, index=False)

    from caas_jupyter_tools import display_dataframe_to_user
    display_dataframe_to_user("GDELT Event Metrics (preview)", metrics_df.head(20))
    display_dataframe_to_user("GDELT Raw Hits (preview)", raw_hits_df.head(20))

    return metrics_path, raw_path

In [12]:
# Show a friendly reminder + create empty output files now (so links exist), but skip running without a real CSV.
empty_metrics = pd.DataFrame(columns=[
    "event_row","event_date","end_date","bank_name_1","bank_name_2","ngos",
    "queries_tried","n_articles","n_articles_with_bank","domains_top","languages_top",
    "top_titles","covered_by_major_outlet","sample_urls"
])
empty_raw = pd.DataFrame(columns=["seendate","domain","language","title","url","event_row","event_date","bank_candidates","ngo_candidates"])
empty_metrics.to_csv("gdelt_event_metrics.csv", index=False)
empty_raw.to_csv("gdelt_raw_hits.csv", index=False)

print("✅ Ready. Update INPUT_CSV with your file path, then run:")
print("metrics_path, raw_path = run_gdelt_pipeline(INPUT_CSV)")
print("Outputs will be saved to:")
print(" - gdelt_event_metrics.csv")
print(" - gdelt_raw_hits.csv")

✅ Ready. Update INPUT_CSV with your file path, then run:
metrics_path, raw_path = run_gdelt_pipeline(INPUT_CSV)
Outputs will be saved to:
 - gdelt_event_metrics.csv
 - gdelt_raw_hits.csv


In [13]:
metrics_path, raw_path = run_gdelt_pipeline(INPUT_CSV)

KeyboardInterrupt: 

--------------------------------------
OLD
--------------------------------------

We decided do scrape **Reddit** to identify if social media talked about the events connected to NGO campaigns.

For this we'll be looking for any posts published in the **10-days period after the campaign occured**, including the name of the **targeted company** and, optionally, the **name of the NGO**.

In [20]:
# -----------------------------
# Config
# -----------------------------
SUBREDDITS = ["all"]  # or e.g., ["finance", "economics", "europe", "ukpolitics"]
MAX_PAGES_PER_QUERY = 10        # safety to avoid infinite loops
REQUESTS_PER_SECOND = 0.1       # be nice to Reddit; adjust if you get rate limited
USER_AGENT = "CampaignRedditScraper/1.0 (by u/YourUsername)"

INPUT_CSV  = "events_date_ngos_company_wide.csv"
OUTPUT_CSV = "reddit_matches_10day_window.csv"

In [22]:
# -----------------------------
# Helpers
# -----------------------------
def clean_name(x):
    """Return a stripped string or None."""
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s else None

def unique_names_from_row(row):
    """Collect present names from the row (company + NGO names)."""
    fields = ["isin_corporate_name_cleaned", "ngo_name1", "ngo_name2", "ngo_name3", "ngo_name4", "ngo_name5", "ngo_name6"]
    vals = [clean_name(row.get(f)) for f in fields if f in row]
    return sorted(set(v for v in vals if v))

def parse_date_ymd(s):
    """CSV shows 'YYYY-MM-DD'—parse to UTC-aware dt."""
    dt = DateTime.strptime(s, "%Y-%m-%d")
    return dt.replace(tzinfo=timezone.utc)

def to_utc_datetime(created_utc):
    """Convert epoch seconds to UTC datetime."""
    return DateTime.fromtimestamp(float(created_utc), tz=timezone.utc)

def search_reddit_for_name(name, start_dt, end_dt, subreddit="all", max_pages=10):
    """
    Use Reddit's search.json. We quote the name to improve precision.
    We paginate via 'after'. We still filter strictly by created_utc in [start_dt, end_dt].
    """
    url = "https://www.reddit.com/search.json"
    headers = {"User-Agent": USER_AGENT}

    # Search query: quoted phrase. If you want broader matches, drop quotes.
    q = f"\"{name}\""
    params = {
        "q": q,
        "sort": "new",
        "limit": 100,
        "t": "all",
    }
    # Restrict to a subreddit if not "all"
    if subreddit.lower() != "all":
        params["restrict_sr"] = 1
        # prepend subreddit: qualifier to improve result quality
        params["q"] = f"subreddit:{subreddit} {q}"

    after = None
    out = []
    pages = 0

    while pages < max_pages:
        if after:
            params["after"] = after

        resp = requests.get(url, headers=headers, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json().get("data", {})
        children = data.get("children", [])
        after = data.get("after")
        pages += 1

        for ch in children:
            d = ch.get("data", {})
            created = to_utc_datetime(d.get("created_utc"))
            if start_dt <= created <= end_dt:
                out.append({
                    "id": d.get("id"),
                    "subreddit": d.get("subreddit"),
                    "title": d.get("title"),
                    "author": d.get("author"),
                    "created_utc": created.isoformat(),
                    "score": d.get("score"),
                    "num_comments": d.get("num_comments"),
                    "url": d.get("url"),
                    "selftext": d.get("selftext", ""),
                    "permalink": f"https://reddit.com{d.get('permalink')}",
                })

        # Heuristic exit: if the newest post on this page is already older than the window start,
        # further pages will likely be even older; break early.
        if children:
            newest_created = to_utc_datetime(children[0]["data"]["created_utc"])
            if newest_created < start_dt:
                break

        if not after:
            break

        time.sleep(1.0 / REQUESTS_PER_SECOND)

    return out

In [23]:
import datetime as _dt
print(type(datetime))                         # should be <class 'type'> if it's the class; <class 'module'> if module
print(getattr(datetime, "__file__", None))    # stdlib module will show a path; the class has no __file__


<class 'module'>
/Users/lucamilani/opt/anaconda3/envs/ima/lib/python3.11/datetime.py


In [24]:
# -----------------------------
# Main
# -----------------------------
df = pd.read_csv(INPUT_CSV)

# Ensure required columns exist
needed = {"date", "isin_corporate_name_cleaned", "ngo_name1", "ngo_name2", "ngo_name3", "ngo_name4", "ngo_name5"}
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns in CSV: {missing}")

results = []
seen_ids = set()  # global de-dup by Reddit post id
cache = {}        # cache[(name, start_iso, end_iso, subreddit)] -> list of posts

for idx, row in df.iterrows():
    # Build 10-day window centered on 'date' (±5 days)
    center = parse_date_ymd(str(row["date"]))
    start_dt = center
    end_dt   = center + timedelta(days=10)

    names = unique_names_from_row(row)
    if not names:
        continue

    for name in names:
        for sr in SUBREDDITS:
            key = (name, start_dt.date().isoformat(), end_dt.date().isoformat(), sr)
            if key not in cache:
                try:
                    cache[key] = search_reddit_for_name(name, start_dt, end_dt, subreddit=sr, max_pages=MAX_PAGES_PER_QUERY)
                except requests.RequestException as e:
                    print(f"[warn] request failed for {key}: {e}")
                    cache[key] = []

            for post in cache[key]:
                if post["id"] in seen_ids:
                    continue
                seen_ids.add(post["id"])

                results.append({
                    # campaign metadata
                    "campaign_row": idx,
                    "campaign_date": center.date().isoformat(),
                    "company": clean_name(row["isin_corporate_name_cleaned"]),
                    "matched_name": name,
                    # reddit fields
                    **post
                })

[warn] request failed for ('Deutsche Bank', '2010-12-23', '2011-01-02', 'all'): 429 Client Error: Too Many Requests for url: https://www.reddit.com/search.json?q=%22Deutsche+Bank%22&sort=new&limit=100&t=all
[warn] request failed for ('Urgewald', '2010-12-23', '2011-01-02', 'all'): 429 Client Error: Too Many Requests for url: https://www.reddit.com/search.json?q=%22Urgewald%22&sort=new&limit=100&t=all
[warn] request failed for ('Commerzbank', '2010-12-23', '2011-01-02', 'all'): 429 Client Error: Too Many Requests for url: https://www.reddit.com/search.json?q=%22Commerzbank%22&sort=new&limit=100&t=all
[warn] request failed for ('UniCredit', '2010-12-23', '2011-01-02', 'all'): 429 Client Error: Too Many Requests for url: https://www.reddit.com/search.json?q=%22UniCredit%22&sort=new&limit=100&t=all
[warn] request failed for ('Greenpeace Norway', '2010-12-22', '2011-01-01', 'all'): 429 Client Error: Too Many Requests for url: https://www.reddit.com/search.json?q=%22Greenpeace+Norway%22&sort

KeyboardInterrupt: 

In [ ]:
# Save
out_df = pd.DataFrame(results)
out_df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(out_df)} posts to {OUTPUT_CSV}")